<a href="https://colab.research.google.com/github/rebeccamanzini/alzheimers-drug-discovery/blob/main/notebooks/ad_transcriptomics_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
setwd("/content")
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
BiocManager::install(setdiff(c("GEOquery", "limma"), rownames(installed.packages())), update = FALSE, ask = FALSE)
install.packages(setdiff(c("ggplot2", "pheatmap"), rownames(installed.packages())))
library(GEOquery); library(limma); library(ggplot2)
for (d in c("data", "results", "figures", "checkpoints")) dir.create(d, showWarnings = FALSE)

f <- "data/GSE5281_series_matrix.txt.gz"
if (!file.exists(f)) download.file(
  "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE5nnn/GSE5281/matrix/GSE5281_series_matrix.txt.gz",
  destfile = f, mode = "wb")
gse <- getGEO(filename = f)
expr_log <- log2(exprs(gse) + 1)   # data are on a linear scale
metadata <- pData(gse)

group <- factor(ifelse(grepl("control", metadata$title, ignore.case = TRUE), "control", "AD"),
                levels = c("control", "AD"))
text_cols <- intersect(c("title", "source_name_ch1",
                         grep("^characteristics", names(metadata), value = TRUE)), names(metadata))
sample_text <- tolower(apply(metadata[, text_cols, drop = FALSE], 1, paste, collapse = " "))
region <- ifelse(grepl("entorhinal", sample_text), "EC",
          ifelse(grepl("hippocamp", sample_text), "HIP",
          ifelse(grepl("temporal", sample_text), "MTG",
          ifelse(grepl("cingulate", sample_text), "PC",
          ifelse(grepl("frontal", sample_text), "SFG",
          ifelse(grepl("visual|occipital", sample_text), "VCX", NA))))))
stopifnot(!any(is.na(region)))
region <- factor(region)
print(table(group)); print(table(region, group))

# Batch check by accession range
num <- as.numeric(sub("GSM", "", colnames(expr_log)))
batch_tab <- table(group, accession_range = paste0("GSM", floor(num / 10000), "xxxx"))
print(batch_tab)
write.csv(as.data.frame.matrix(batch_tab), "results/batch_confounding_by_accession_range.csv")

# QC and outlier arrays
png("figures/qc_sample_distributions.png", width = 1400, height = 600, res = 150)
boxplot(expr_log, outline = FALSE, names = rep("", ncol(expr_log)), xlab = "Samples",
        ylab = "log2(intensity + 1)", main = "Sample distributions after log2 transform")
dev.off()
arr_med <- apply(expr_log, 2, median)
arr_q75 <- apply(expr_log, 2, quantile, probs = 0.75)
rz <- function(x) (x - median(x)) / mad(x)
flag <- abs(rz(arr_med)) > 4 | abs(rz(arr_q75)) > 4
cat("Flagged arrays:", sum(flag), "\n")
write.csv(data.frame(sample = colnames(expr_log)[flag], group = group[flag], region = region[flag]),
          "results/qc_flagged_arrays.csv", row.names = FALSE)

# Control probes, spike-in technical score, PCA
affx <- grepl("^AFFX", rownames(expr_log))
spk <- grepl("^AFFX-(r2-Bs-|DapX|LysX|PheX|ThrX)", rownames(expr_log))
tech <- as.numeric(scale(prcomp(t(expr_log[spk, ]), scale. = TRUE)$x[, 1]))
cat("Control probes:", sum(affx), " spike-in probes used:", sum(spk), "\n")
cat("Technical score by group\n"); print(round(tapply(tech, group, mean), 2))
cat("R2 with group:", round(summary(lm(tech ~ group))$r.squared, 2),
    " with region:", round(summary(lm(tech ~ region))$r.squared, 2), "\n")
p_tech <- ggplot(data.frame(tech = tech, group = group, region = region), aes(group, tech, colour = region)) +
  geom_boxplot(outlier.shape = NA) + geom_jitter(width = 0.15, alpha = 0.6) +
  labs(title = "Spike-in technical score by diagnosis", x = "", y = "Technical score (PC1 of spike-in probes)") +
  theme_minimal()
ggsave("figures/spikein_technical_score_by_group.png", plot = p_tech, width = 7, height = 5, dpi = 300)

expr_vis <- expr_log[!affx, ]
expr_vis <- expr_vis[rowMeans(expr_vis) > quantile(rowMeans(expr_vis), 0.5), ]
pca <- prcomp(t(expr_vis), scale. = TRUE)
var_exp <- round(100 * pca$sdev^2 / sum(pca$sdev^2), 1)
p_pca <- ggplot(data.frame(PC1 = pca$x[, 1], PC2 = pca$x[, 2], group = group, region = region),
                aes(PC1, PC2, colour = region, shape = group)) +
  geom_point(size = 2.5, alpha = 0.8) +
  labs(title = "PCA of GSE5281 samples", x = paste0("PC1 (", var_exp[1], "%)"),
       y = paste0("PC2 (", var_exp[2], "%)")) + theme_minimal()
ggsave("figures/pca_by_region_and_group.png", plot = p_pca, width = 7, height = 5, dpi = 300)

# Models
lc <- log2(1.5)
meta_k <- droplevels(data.frame(group = group, region = region, tech = tech)[!flag, ])
expr_Q <- normalizeBetweenArrays(expr_log[!affx, !flag], method = "quantile")
expr_f <- expr_Q[rowMeans(expr_Q) > quantile(rowMeans(expr_Q), 0.5), ]
cat("Arrays used:", ncol(expr_f), " probes tested:", nrow(expr_f), "\n")
fit0 <- lmFit(expr_f, model.matrix(~ region + group, data = meta_k))
fit1 <- lmFit(expr_f, model.matrix(~ region + tech + group, data = meta_k))
res_P  <- topTreat(treat(fit0, lfc = lc), coef = "groupAD", number = Inf, sort.by = "P")
res_S1 <- topTreat(treat(fit1, lfc = lc), coef = "groupAD", number = Inf, sort.by = "P")
tt0 <- topTable(eBayes(fit0), coef = "groupAD", number = Inf, sort.by = "none")
tt1 <- topTable(eBayes(fit1), coef = "groupAD", number = Inf, sort.by = "none")
res_A <- topTable(eBayes(lmFit(expr_log, model.matrix(~ group, data = data.frame(group = group)))),
                  coef = "groupAD", number = Inf, sort.by = "P")

cnt <- function(tt, lfc = 0) sum(tt$adj.P.Val < 0.05 & abs(tt$logFC) > lfc)
sens <- data.frame(
  model = c("Original analysis (group only, all probes and arrays, abs log2FC > 1)",
            "Primary (region-adjusted, control probes and 8 flagged arrays removed, quantile normalised, treat vs 1.5-fold)",
            "Primary, ordinary test, no fold-change threshold",
            "Primary, ordinary test, abs log2FC > log2(1.5)",
            "Sensitivity, plus spike-in technical covariate, treat vs 1.5-fold",
            "Sensitivity, plus technical covariate, ordinary test, abs log2FC > log2(1.5)",
            "Sensitivity, plus technical covariate, ordinary test, no threshold"),
  n_significant = c(cnt(res_A, 1), sum(res_P$adj.P.Val < 0.05), cnt(tt0), cnt(tt0, lc),
                    sum(res_S1$adj.P.Val < 0.05), cnt(tt1, lc), cnt(tt1)))
write.csv(sens, "results/sensitivity_summary.csv", row.names = FALSE)
print(sens)
sig_P <- res_P[res_P$adj.P.Val < 0.05, ]
cat("Primary higher in AD:", sum(sig_P$logFC > 0), " lower:", sum(sig_P$logFC < 0), "\n")
cat("Spearman correlation of moderated t, primary vs covariate-adjusted:",
    round(cor(tt0$t, tt1$t, method = "spearman"), 3), "\n")
cat("Slope of adjusted log fold change on primary:", round(coef(lm(tt1$logFC ~ tt0$logFC))[2], 2), "\n")

saveRDS(list(expr_f = expr_f, meta_k = meta_k, res_P = res_P, res_S1 = res_S1, tt0 = tt0, tt1 = tt1),
        "checkpoints/models.rds")
cat("Cell 1 finished\n")

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.1 (2026-06-24)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Using locally cached version of GPL570 found here:
/tmp/RtmpcWVMBQ/GPL570.soft.gz 



group
control      AD 
     74      87 
      group
region control AD
   EC       13 10
   HIP      13 10
   MTG      12 16
   PC       13  9
   SFG      11 23
   VCX      12 19
         accession_range
group     GSM11xxxx GSM23xxxx
  control        74         0
  AD              0        87


agg_record_ff4470348c6 
                     2

Flagged arrays: 8 
Control probes: 62  spike-in probes used: 24 
Technical score by group
control      AD 
   0.60   -0.51 
R2 with group: 0.31  with region: 0.38 
Arrays used: 153  probes tested: 27306 
                                                                                                           model
1                                          Original analysis (group only, all probes and arrays, abs log2FC > 1)
2 Primary (region-adjusted, control probes and 8 flagged arrays removed, quantile normalised, treat vs 1.5-fold)
3                                                               Primary, ordinary test, no fold-change threshold
4                                                                 Primary, ordinary test, abs log2FC > log2(1.5)
5                                              Sensitivity, plus spike-in technical covariate, treat vs 1.5-fold
6                                   Sensitivity, plus technical covariate, ordinary test, abs log2FC > log2(1.5)
7    

In [6]:
setwd("/content")
BiocManager::install(setdiff(c("clusterProfiler", "org.Hs.eg.db"), rownames(installed.packages())),
                     update = FALSE, ask = FALSE)
library(GEOquery); library(ggplot2); library(pheatmap); library(clusterProfiler); library(org.Hs.eg.db)
m <- readRDS("checkpoints/models.rds")
expr_f <- m$expr_f; meta_k <- m$meta_k; res_P <- m$res_P; tt1 <- m$tt1
lc <- log2(1.5)

gpl <- getGEO("GPL570", AnnotGPL = TRUE)
platform_data <- Table(gpl)[, c("ID", "Gene symbol")]; rm(gpl); gc()
id_to_symbol <- setNames(sub("\\s*///.*$", "", platform_data$`Gene symbol`), platform_data$ID)
saveRDS(id_to_symbol, "checkpoints/id_to_symbol.rds")

sig_P <- res_P[res_P$adj.P.Val < 0.05, ]; sig_P$gene_symbol <- id_to_symbol[rownames(sig_P)]
sig_S2 <- tt1[tt1$adj.P.Val < 0.05 & abs(tt1$logFC) > lc, ]; sig_S2$gene_symbol <- id_to_symbol[rownames(sig_S2)]
write.csv(res_P, "results/de_results_primary_all.csv")
write.csv(sig_P, "results/sig_genes_primary.csv")
write.csv(sig_S2, "results/sig_genes_sensitivity_technical_covariate.csv")

# Volcano
res_P$significant <- ifelse(res_P$adj.P.Val < 0.05, "Significant", "Not significant")
p_volcano <- ggplot(res_P, aes(x = logFC, y = -log10(P.Value), colour = significant)) +
  geom_point(alpha = 0.5) +
  scale_colour_manual(values = c("Significant" = "red", "Not significant" = "grey")) +
  labs(title = "AD vs control (GSE5281, region-adjusted, 153 arrays)",
       x = "log2 fold change (AD relative to control)", y = "-log10 p-value", colour = "") +
  theme_minimal()
ggsave("figures/volcano_plot_AD_vs_control.png", plot = p_volcano, width = 7, height = 5, dpi = 300)

# Heatmap, top 30 probes
top30 <- rownames(sig_P)[1:min(30, nrow(sig_P))]
heat <- expr_f[top30, ]
lab <- id_to_symbol[top30]
rownames(heat) <- make.unique(ifelse(is.na(lab) | lab == "", top30, lab))
ann <- data.frame(Group = meta_k$group, Region = meta_k$region); rownames(ann) <- colnames(expr_f)
pheatmap(heat, annotation_col = ann, show_colnames = FALSE, scale = "row",
         main = "Top 30 differentially expressed probes, AD vs control",
         filename = "figures/heatmap_top30_genes.png", width = 9, height = 7)

# GO enrichment, background = genes on the array
universe <- unique(id_to_symbol[rownames(expr_f)])
universe <- universe[!is.na(universe) & universe != ""]
run_go <- function(genes, label, title) {
  genes <- unique(genes[!is.na(genes) & genes != ""])
  if (length(genes) < 10) { message("Too few genes for ", label); return(NULL) }
  ego <- enrichGO(gene = genes, universe = universe, OrgDb = org.Hs.eg.db, keyType = "SYMBOL",
                  ont = "BP", pAdjustMethod = "BH", pvalueCutoff = 0.05, qvalueCutoff = 0.05)
  if (is.null(ego) || nrow(as.data.frame(ego)) == 0) { message("No enriched terms for ", label); return(NULL) }
  write.csv(as.data.frame(ego), paste0("results/go_enrichment_", label, ".csv"), row.names = FALSE)
  p <- dotplot(ego, showCategory = 12, label_format = 35) + ggtitle(title)
  ggsave(paste0("figures/go_enrichment_dotplot_", label, ".png"), plot = p, width = 9, height = 8, dpi = 300)
  ego
}
go_list <- list(
  lower_in_AD = run_go(sig_P$gene_symbol[sig_P$logFC < 0], "lower_in_AD", "Lower in AD (primary)"),
  higher_in_AD = run_go(sig_P$gene_symbol[sig_P$logFC > 0], "higher_in_AD", "Higher in AD (primary)"),
  lower_in_AD_adjusted = run_go(sig_S2$gene_symbol[sig_S2$logFC < 0], "lower_in_AD_technical_adjusted", "Lower in AD (technical covariate)"),
  higher_in_AD_adjusted = run_go(sig_S2$gene_symbol[sig_S2$logFC > 0], "higher_in_AD_technical_adjusted", "Higher in AD (technical covariate)"))
for (nm in names(go_list)) {
  if (!is.null(go_list[[nm]])) { cat("\n", nm, "top terms\n"); print(head(as.data.frame(go_list[[nm]])[, c("Description", "p.adjust")], 6)) }
}

# Gene list for drug mining
gene_list <- data.frame(gene = sig_P$gene_symbol, logFC = sig_P$logFC, adj.P.Val = sig_P$adj.P.Val)
gene_list <- gene_list[!is.na(gene_list$gene) & gene_list$gene != "", ]
gene_list <- gene_list[!duplicated(gene_list$gene), ]
write.csv(gene_list, "results/gene_list_for_drug_mining.csv", row.names = FALSE)
cat("\nPrimary significant probes:", nrow(sig_P), " unique genes in drug-mining list:", nrow(gene_list), "\n")
writeLines(capture.output(sessionInfo()), "results/sessionInfo.txt")
cat("Cell 2 finished\n")

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.1 (2026-06-24)

Installing package(s) 'clusterProfiler', 'org.Hs.eg.db'

also installing the dependencies ‘fontBitstreamVera’, ‘fontLiberation’, ‘fontquiver’, ‘gridGraphics’, ‘gdtools’, ‘lazyeval’, ‘tweenr’, ‘polyclip’, ‘png’, ‘Biostrings’, ‘ggplotify’, ‘patchwork’, ‘ape’, ‘ggiraph’, ‘tidytree’, ‘treeio’, ‘ggforce’, ‘RSQLite’, ‘KEGGREST’, ‘RcppEigen’, ‘aplot’, ‘DOSE’, ‘ggfun’, ‘ggnewscale’, ‘ggrepel’, ‘ggtangle’, ‘ggtree’, ‘reshape2’, ‘scatterpie’, ‘tidydr’, ‘aisdk’, ‘AnnotationDbi’, ‘enrichit’, ‘enrichplot’, ‘GO.db’, ‘GOSemSim’, ‘gson’, ‘igraph’, ‘plyr’, ‘qvalue’, ‘yulab.utils’




clusterProfiler v4.20.3 Learn more at https://yulab-smu.top/contribution-knowledge-mining/

Please cite:

Guangchuang Yu, Li-Gen Wang, Yanyan Han and Qing-Yu He.


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,9303876,496.9,17978517,960.2,12034445,642.8
Vcells,78132523,596.2,155407442,1185.7,155407441,1185.7


'select()' returned 1:1 mapping between keys and columns

Warning message in bitr(gene, fromType = fromType, toType = "ENTREZID", OrgDb = OrgDb):
“6.35% of input gene IDs are fail to map...”
'select()' returned 1:many mapping between keys and columns

Warning message in bitr(gene, fromType = fromType, toType = "ENTREZID", OrgDb = OrgDb):
“8.92% of input gene IDs are fail to map...”
'select()' returned 1:1 mapping between keys and columns

Warning message in bitr(gene, fromType = fromType, toType = "ENTREZID", OrgDb = OrgDb):
“7.67% of input gene IDs are fail to map...”
'select()' returned 1:many mapping between keys and columns

Warning message in bitr(gene, fromType = fromType, toType = "ENTREZID", OrgDb = OrgDb):
“8.92% of input gene IDs are fail to map...”
'select()' returned 1:1 mapping between keys and columns

Warning message in bitr(gene, fromType = fromType, toType = "ENTREZID", OrgDb = OrgDb):
“7.91% of input gene IDs are fail to map...”
'select()' returned 1:many mapping betw


 lower_in_AD top terms
                                                      Description     p.adjust
GO:0009060                                    aerobic respiration 1.295082e-08
GO:0045333                                   cellular respiration 1.397049e-07
GO:0099003                  vesicle-mediated transport in synapse 5.504653e-07
GO:0006091         generation of precursor metabolites and energy 1.713208e-06
GO:0042773               ATP synthesis coupled electron transport 1.801710e-06
GO:0042775 mitochondrial ATP synthesis coupled electron transport 1.801710e-06

 higher_in_AD top terms
                                           Description    p.adjust
GO:0048729                        tissue morphogenesis 0.008466298
GO:0045667    regulation of osteoblast differentiation 0.008466298
GO:0030855             epithelial cell differentiation 0.008466298
GO:0045596 negative regulation of cell differentiation 0.008963359
GO:0001501                 skeletal system development 0.008963

In [7]:
setwd("/content")
library(GEOquery); library(limma)
py <- r"---(
import datetime
import pandas as pd
import requests

genes_df = pd.read_csv("results/gene_list_for_drug_mining.csv")
top = genes_df.head(50)
genes_top50 = top["gene"].tolist()
url = "https://dgidb.org/api/graphql"
query = """
query genes($names: [String!]) {
  genes(names: $names) {
    nodes {
      name
      interactions {
        drug { name }
        interactionScore
        interactionTypes { type }
      }
    }
  }
}
"""
response = requests.post(url, json={"query": query, "variables": {"names": genes_top50}}, timeout=30)
response.raise_for_status()
data = response.json()
if "errors" in data:
    raise RuntimeError(data["errors"])
nodes = data["data"]["genes"]["nodes"]
rows = []
for node in nodes:
    for inter in node["interactions"]:
        types = ", ".join(t["type"] for t in inter["interactionTypes"]) if inter["interactionTypes"] else "unspecified"
        rows.append({"gene": node["name"], "drug": inter["drug"]["name"],
                     "interaction_type": types, "score": inter["interactionScore"]})
df = pd.DataFrame(rows)
df = df.merge(top[["gene", "logFC", "adj.P.Val"]], on="gene", how="left")
df["direction_in_AD"] = df["logFC"].apply(lambda x: "higher in AD" if x > 0 else "lower in AD")
df = df.sort_values(by="score", ascending=False)
df.to_csv("results/drug_gene_interactions.csv", index=False)
not_found = sorted(set(genes_top50) - {n["name"] for n in nodes})
print("Queried DGIdb on", datetime.date.today())
print("Total interactions found:", len(df))
print("Genes with at least one known drug interaction:", df["gene"].nunique())
print("Genes not found in DGIdb:", not_found)
)---"
writeLines(py, "scripts_drug_target_mining.py")
cat(system("python3 scripts_drug_target_mining.py 2>&1", intern = TRUE), sep = "\n")

dd <- read.csv("results/drug_gene_interactions.csv")
inhib <- grepl("inhibitor|antagonist|blocker|antibody|negative modulator|suppressor", dd$interaction_type, ignore.case = TRUE)
activ <- grepl("agonist|activator|positive modulator|inducer|potentiator", dd$interaction_type, ignore.case = TRUE)
dd$compatible <- (inhib & dd$direction_in_AD == "higher in AD") | (activ & dd$direction_in_AD == "lower in AD")
comp <- dd[dd$compatible, ]; comp <- comp[order(-comp$score), ]
write.csv(comp, "results/drug_gene_direction_compatible.csv", row.names = FALSE)
cat("Direction-compatible interactions:", nrow(comp), "across", length(unique(comp$gene)), "genes\n")
print(sort(unique(comp$gene)))

# Blood comparison
m <- readRDS("checkpoints/models.rds"); id_to_symbol <- readRDS("checkpoints/id_to_symbol.rds")
sig_P_genes <- unique(id_to_symbol[rownames(m$res_P)[m$res_P$adj.P.Val < 0.05]])
collapse_gene <- function(res, symbols) {
  res$gene <- symbols[rownames(res)]
  res <- res[!is.na(res$gene) & res$gene != "", ]
  res <- res[order(res$P.Value), ]
  res[!duplicated(res$gene), ]
}
gse2 <- getGEO("GSE63060", GSEMatrix = TRUE)
expr2 <- exprs(gse2[[1]]); meta2 <- pData(gse2[[1]]); rm(gse2)
k2 <- meta2$`status:ch1` %in% c("AD", "CTL")
expr2 <- expr2[, k2]; meta2 <- meta2[k2, ]
grp2 <- factor(ifelse(meta2$`status:ch1` == "AD", "AD", "control"), levels = c("control", "AD"))
res2 <- topTable(eBayes(lmFit(expr2, model.matrix(~ grp2))), coef = 2, number = Inf, sort.by = "P")
gpl2 <- getGEO("GPL6947", AnnotGPL = TRUE)
p6947 <- Table(gpl2)[, c("ID", "Gene symbol")]; rm(gpl2)
sym6947 <- setNames(sub("\\s*///.*$", "", p6947$`Gene symbol`), p6947$ID)
blood <- collapse_gene(res2, sym6947)

compare <- function(tt, label) {
  b <- collapse_gene(tt, id_to_symbol)
  common <- intersect(b$gene, blood$gene)
  bb <- b[match(common, b$gene), ]; bl <- blood[match(common, blood$gene), ]
  ct <- cor.test(bb$t, bl$t, method = "spearman", exact = FALSE)
  sig <- bb$gene %in% sig_P_genes
  same <- sign(bb$logFC) == sign(bl$logFC)
  bt <- binom.test(sum(same[sig]), sum(sig), 0.5)
  paste0(label, ". Blood arrays ", ncol(expr2), " (AD ", sum(grp2 == "AD"), ", control ", sum(grp2 == "control"),
         "). Shared genes ", length(common), ". Spearman rho (t) ", round(ct$estimate, 3),
         " (p ", signif(ct$p.value, 3), "). Primary-significant genes ", sum(sig),
         ", same direction in blood ", sum(same[sig]), " (", round(100 * mean(same[sig]), 1),
         "%), binomial p ", signif(bt$p.value, 3))
}
lines <- c(compare(m$tt0, "Primary"), compare(m$tt1, "Technical-covariate adjusted"))
writeLines(lines, "results/validation_summary.txt"); cat(lines, sep = "\n")

system("cd /content && zip -r outputs_final.zip results figures")
cat("Cell 3 finished, now download outputs_final.zip\n")

Queried DGIdb on 2026-10-25
Total interactions found: 457
Genes with at least one known drug interaction: 20
Genes not found in DGIdb: ['AMER2', 'ANKRD36', 'ANP32B', 'ATP5C1', 'C1orf61', 'CCDC152', 'FBXO32', 'LOC100506248', 'LOC106146153', 'LOC202181', 'LOC441155', 'MIR612', 'NSUN6', 'PNISR', 'SEC62', 'SRRM2', 'TNPO1']
Direction-compatible interactions: 30 across 9 genes
[1] "ACACB"  "CDK13"  "CFLAR"  "ITPKB"  "MAP4K4" "PGF"    "SLC5A3" "THRA"  
[9] "WNK1"  


Found 1 file(s)

GSE63060_series_matrix.txt.gz

Using locally cached version: /tmp/RtmpcWVMBQ/GSE63060_series_matrix.txt.gz

Using locally cached version of GPL6947 found here:
/tmp/RtmpcWVMBQ/GPL6947.soft.gz 

Using locally cached version of GPL6947 found here:
/tmp/RtmpcWVMBQ/GPL6947.annot.gz 



Primary. Blood arrays 249 (AD 145, control 104). Shared genes 11765. Spearman rho (t) 0.158 (p 1.62e-66). Primary-significant genes 1411, same direction in blood 794 (56.3%), binomial p 2.72e-06
Technical-covariate adjusted. Blood arrays 249 (AD 145, control 104). Shared genes 11765. Spearman rho (t) 0.094 (p 1.06e-24). Primary-significant genes 1411, same direction in blood 800 (56.7%), binomial p 5.39e-07
Cell 3 finished, now download outputs_final.zip
